In [1]:
import pandas as pd
from pathlib import Path
import os

In [2]:
# Create required folders
os.makedirs('data/stage-1', exist_ok=True)
os.makedirs('data/stage-2', exist_ok=True)
os.makedirs('data/stage-3', exist_ok=True)

Combine Meteorological Time Series Dataset
Reads all variable subfolders (Irradiance, Rainfall, Relative Humidity,
Sea Level Pressure, Temperature, Visibility, Wind), concatenates the
yearly files (2021-2023) within each variable, then inner-joins all
7 variables together on a common timestamp column into one combined
DataFrame.

Handles mixed file formats (.csv and .xlsx) automatically.

In [2]:
ROOT = Path("data/Dataset/Time series dataset/Meteorological dataset")

VARIABLES = [
    "Irradiance",
    "Rainfall",
    "Relative Humidity",
    "Sea Level Pressure",
    "Temperature",
    "Visibility",
    "Wind",
]

# Candidate names for the datetime column — extend if your files differ
DATETIME_CANDIDATES = [
    "datetime", "date_time", "timestamp", "date", "Date", "DATE",
    "Datetime", "DateTime", "Timestamp", "DATE_TIME", "time", "Time"
]

In [3]:



# ---------------------------------------------------------------------
# 2. HELPERS
# ---------------------------------------------------------------------
def read_any(path: Path) -> pd.DataFrame:
    """Read a .csv or .xlsx file into a DataFrame."""
    if path.suffix.lower() == ".xlsx":
        return pd.read_excel(path)
    return pd.read_csv(path)


def find_datetime_col(df: pd.DataFrame) -> str:
    """Find the datetime column in a DataFrame by name match."""
    for col in df.columns:
        if col.strip() in DATETIME_CANDIDATES or col.strip().lower() in [
            c.lower() for c in DATETIME_CANDIDATES
        ]:
            return col
    raise ValueError(
        f"No datetime column found. Columns present: {list(df.columns)}"
    )


def load_variable(var_folder: Path, var_name: str) -> pd.DataFrame:
    """Load & concatenate all yearly files for one variable folder."""
    files = sorted(
        [f for f in var_folder.iterdir() if f.suffix.lower() in (".csv", ".xlsx")]
    )
    if not files:
        raise FileNotFoundError(f"No CSV/XLSX files found in {var_folder}")

    yearly_frames = []
    for f in files:
        df = read_any(f)
        df.columns = [c.strip() for c in df.columns]  # clean column names
        dt_col = find_datetime_col(df)
        df = df.rename(columns={dt_col: "datetime"})
        df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
        df = df.dropna(subset=["datetime"])

        # Rename all non-datetime columns to be prefixed with the variable
        # name, so merging doesn't create column collisions.
        value_cols = [c for c in df.columns if c != "datetime"]
        rename_map = {c: f"{var_name}_{c}" for c in value_cols}
        df = df.rename(columns=rename_map)

        yearly_frames.append(df)
        print(f"  Loaded {f.name}: {df.shape[0]} rows")

    combined = pd.concat(yearly_frames, ignore_index=True)
    combined = combined.sort_values("datetime").drop_duplicates(subset="datetime")
    print(f"-> {var_name}: {combined.shape[0]} total rows after concat\n")
    return combined


# ---------------------------------------------------------------------
# 3. LOAD EACH VARIABLE FOLDER
# ---------------------------------------------------------------------
variable_frames = {}

for var in VARIABLES:
    folder = ROOT / var
    print(f"Processing '{var}' from {folder} ...")
    variable_frames[var] = load_variable(folder, var.replace(" ", "_"))


# ---------------------------------------------------------------------
# 4. INNER-JOIN ALL VARIABLES ON DATETIME
# ---------------------------------------------------------------------
merged = None
for var, df in variable_frames.items():
    if merged is None:
        merged = df
    else:
        merged = pd.merge(merged, df, on="datetime", how="inner")

merged = merged.sort_values("datetime").reset_index(drop=True)

print(f"Final combined dataset shape: {merged.shape}")
print(merged.head())


# ---------------------------------------------------------------------
# 5. SAVE OUTPUT
# ---------------------------------------------------------------------
OUTPUT_PATH = "data/stage-1/merged_meteorological_data.csv"
merged.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved combined dataset to: {OUTPUT_PATH}")

Processing 'Irradiance' from data/Dataset/Time series dataset/Meteorological dataset/Irradiance ...
  Loaded Irradiance_2021.csv: 525598 rows
  Loaded Irradiance_2022.csv: 522524 rows
  Loaded Irradiance_2023.csv: 519670 rows
-> Irradiance: 1567792 total rows after concat

Processing 'Rainfall' from data/Dataset/Time series dataset/Meteorological dataset/Rainfall ...
  Loaded Rainfall_2021.xlsx: 525598 rows
  Loaded Rainfall_2022.csv: 522524 rows
  Loaded Rainfall_2023.csv: 519670 rows
-> Rainfall: 1567792 total rows after concat

Processing 'Relative Humidity' from data/Dataset/Time series dataset/Meteorological dataset/Relative Humidity ...
  Loaded Relative Humidity_2021.csv: 525598 rows
  Loaded Relative Humidity_2022.csv: 522397 rows
  Loaded Relative Humidity_2023.csv: 519670 rows
-> Relative_Humidity: 1567665 total rows after concat

Processing 'Sea Level Pressure' from data/Dataset/Time series dataset/Meteorological dataset/Sea Level Pressure ...
  Loaded Sea Level Pressure_202